# Machine Learning-Based Predictive Maintenance and Machine Failure Risk Prediction System
TE Machine Learning Mini Project. Upload `ai4i2020.csv` to Colab before running.

## 1. Problem definition
Binary classification: predict `Machine failure` (0 = no failure, 1 = failure) from five readings.

In [ ]:
import pandas as pd, numpy as np, pickle, json
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Dataset

In [ ]:
df = pd.read_csv('ai4i2020.csv')
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe()

## 3. Preprocessing

In [ ]:
FEATURES = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
TARGET = 'Machine failure'
df = df[FEATURES + [TARGET]]
print('Missing values:\n', df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
df = df.dropna().drop_duplicates()
df.to_csv('prepared_predictive_maintenance.csv', index=False)
df.shape

No categorical features are used, so no encoding is needed. Scaling is done with `StandardScaler` inside the model pipeline.

## 4. Exploratory data analysis

In [ ]:
sns.countplot(x=TARGET, data=df); plt.title('Failure vs no failure'); plt.show()
df[TARGET].value_counts(normalize=True)

In [ ]:
df[FEATURES].hist(figsize=(12,7), bins=30); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,5)); sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm'); plt.show()

In [ ]:
fig, axes = plt.subplots(1,5, figsize=(20,4))
for ax, c in zip(axes, FEATURES):
    sns.boxplot(data=df, x=TARGET, y=c, ax=ax)
plt.tight_layout(); plt.show()

## 5. Model training

In [ ]:
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {
 'Logistic Regression': Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000, class_weight='balanced'))]),
 'Random Forest': Pipeline([('scaler',StandardScaler()),('model',RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))]),
}
for m in models.values(): m.fit(X_train, y_train)

## 6. Evaluation

In [ ]:
metrics = {}
for name, pipe in models.items():
    pred = pipe.predict(X_test)
    metrics[name] = {'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),
        'Recall':recall_score(y_test,pred,zero_division=0),'F1-Score':f1_score(y_test,pred,zero_division=0),
        'Confusion Matrix':confusion_matrix(y_test,pred).tolist()}
    print(name); print(classification_report(y_test, pred, zero_division=0))
    sns.heatmap(confusion_matrix(y_test,pred), annot=True, fmt='d', cmap='Blues'); plt.title(name); plt.show()
best = max(metrics, key=lambda m: metrics[m]['F1-Score'])
print('Best model:', best)
pd.DataFrame({k:{a:b for a,b in v.items() if a!='Confusion Matrix'} for k,v in metrics.items()}).T

## 7. Save files for the Streamlit app

In [ ]:
metrics['_best_model'] = best
metrics['_rows'] = {'total':int(len(df)),'train':int(len(X_train)),'test':int(len(X_test))}
pickle.dump(models['Logistic Regression'], open('logistic_regression_model.pkl','wb'))
pickle.dump(models['Random Forest'], open('random_forest_model.pkl','wb'))
pickle.dump(FEATURES, open('feature_columns.pkl','wb'))
json.dump(metrics, open('metrics.json','w'), indent=2)

## 8. Conclusion
Write your results, limitations (synthetic, imbalanced data; manual input) and future scope here after running.